**Loading all the Necessary Libraries for this Notebook**

In [ ]:
# Import Necessary Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.3f}')
sns.set(style='whitegrid')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv("/content/drive/MyDrive/supply_chain_thesis/supply_chain_dataset1.csv")
display(df.head(), df.tail())

Mounted at /content/drive
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Date,SKU_ID,Warehouse_ID,Supplier_ID,Region,Units_Sold,Inventory_Level,Supplier_Lead_Time_Days,Reorder_Point,Order_Quantity,Unit_Cost,Unit_Price,Promotion_Flag,Stockout_Flag,Demand_Forecast
0,2024-01-01,SKU_1,WH_1,SUP_8,West,10,592,14,379,0,13.950,20.480,0,0,8.520
1,2024-01-02,SKU_1,WH_1,SUP_8,West,17,575,14,379,0,13.950,20.480,0,0,18.630
2,2024-01-03,SKU_1,WH_1,SUP_8,North,35,540,14,379,0,13.950,20.480,1,0,39.620
3,2024-01-04,SKU_1,WH_1,SUP_8,South,24,516,14,379,0,13.950,20.480,0,0,19.430
4,2024-01-05,SKU_1,WH_1,SUP_8,West,21,495,14,379,0,13.950,20.480,0,0,18.700


,Date,SKU_ID,Warehouse_ID,Supplier_ID,Region,Units_Sold,Inventory_Level,Supplier_Lead_Time_Days,Reorder_Point,Order_Quantity,Unit_Cost,Unit_Price,Promotion_Flag,Stockout_Flag,Demand_Forecast
91245,2024-12-26,SKU_50,WH_5,SUP_10,South,17,352,7,283,0,6.650,9.600,0,0,19.820
91246,2024-12-27,SKU_50,WH_5,SUP_10,South,21,331,7,283,0,6.650,9.600,0,0,27.960
91247,2024-12-28,SKU_50,WH_5,SUP_10,East,17,314,7,283,0,6.650,9.600,0,0,22.130
91248,2024-12-29,SKU_50,WH_5,SUP_10,South,24,290,7,283,0,6.650,9.600,1,0,24.110
91249,2024-12-30,SKU_50,WH_5,SUP_10,West,15,275,7,283,0,6.650,9.600,1,0,20.190


 - **Temporal features**

In [ ]:

# Ensure Date is datetime
df['Date'] = pd.to_datetime(df['Date'])

# Temporal features
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Day'] = df['Date'].dt.day
df['Day_Of_Week'] = df['Date'].dt.dayofweek
df['Week_Of_Year'] = df['Date'].dt.isocalendar().week.astype(int)
df['Quarter'] = df['Date'].dt.quarter

This where features extract calendar information from the Date column, so that the **models** learn seasonality and calendar-driven demand patterns

 - **Lag features**

In [ ]:
# Ensure the dataframe is sorted by date
df = df.sort_values('Date')

# Lag features
df['Units_Sold_lag1'] = df['Units_Sold'].shift(1)
df['Units_Sold_lag7'] = df['Units_Sold'].shift(7)
df['Units_Sold_lag30'] = df['Units_Sold'].shift(30)


this Lag Features ilustrates the  past demand is the strongest predictor of future demand.

 - **Rolling window features**

In [ ]:
# Rolling window features
df['Units_Sold_roll7_mean'] = df['Units_Sold'].rolling(window=7).mean()
df['Units_Sold_roll30_mean'] = df['Units_Sold'].rolling(window=30).mean()
df['Units_Sold_roll7_std'] = df['Units_Sold'].rolling(window=7).std()


 - **Promotion Interaction Features**

In [ ]:
df['Promo_Effect'] = df['Promotion_Flag'] * df['Units_Sold']
df['Promo_Last_7_Days'] = df['Promotion_Flag'].rolling(window=7).sum()
df['Promo_Roll7_Mean'] = df['Promotion_Flag'].rolling(window=7).mean()


- **Trend‑based regression features**

In [ ]:

# Trend-based Regression Features Weekly and Monthly

from sklearn.linear_model import LinearRegression
import numpy as np

def rolling_regression_slope(series, window):
    slopes = []
    for i in range(len(series)):
        if i < window:
            slopes.append(np.nan)
        else:
            y = series[i-window:i].values.reshape(-1, 1)
            x = np.arange(window).reshape(-1, 1)
            model = LinearRegression().fit(x, y)
            slopes.append(model.coef_[0][0])
    return slopes

# Weekly trend (7-day slope)
df['weekly_slope'] = rolling_regression_slope(df['Units_Sold'], 7)

# Monthly trend (30-day slope)
df['monthly_slope'] = rolling_regression_slope(df['Units_Sold'], 30)

df[['weekly_slope', 'monthly_slope']].head(15)


,weekly_slope,monthly_slope
0,NaN,NaN
10585,NaN,NaN
68255,NaN,NaN
68620,NaN,NaN
10220,NaN,NaN
68985,NaN,NaN
69350,NaN,NaN
69715,-0.536,NaN
9855,-2.250,NaN
70080,0.214,NaN


The `Weekly slope and Monthly slope` feature is simply a numerical description of the **recent direction of demand**. It is a prediction with summary statistic extracted from the last few days.

- **Preparing SKU_18 Feature Matrix**

In [ ]:
df_sku18_fe = df[df['SKU_ID'] == 'SKU_18'].copy()

# Preprocessing
 - **One-Hot Encoding to Categorical Columns**

Applying one-hot encoding to the categorical columns: 'SKU_ID', 'Warehouse_ID', 'Supplier_ID', and 'Region'. Ensure that the original categorical columns are dropped after encoding to avoid redundancy and collinearity issues.


In [ ]:
categorical_cols = ['SKU_ID', 'Warehouse_ID', 'Supplier_ID', 'Region']
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True, dtype=int)
print("One-hot encoding applied to categorical columns. New DataFrame shape:", df.shape)
display(df.head())

One-hot encoding applied to categorical columns. New DataFrame shape: (91250, 93)


,Date,Units_Sold,Inventory_Level,Supplier_Lead_Time_Days,Reorder_Point,Order_Quantity,Unit_Cost,Unit_Price,Promotion_Flag,Stockout_Flag,Demand_Forecast,Year,Month,Day,Day_Of_Week,Week_Of_Year,Quarter,Units_Sold_lag1,Units_Sold_lag7,Units_Sold_lag30,Units_Sold_roll7_mean,Units_Sold_roll30_mean,Units_Sold_roll7_std,Promo_Effect,Promo_Last_7_Days,Promo_Roll7_Mean,weekly_slope,monthly_slope,SKU_ID_SKU_10,SKU_ID_SKU_11,SKU_ID_SKU_12,SKU_ID_SKU_13,SKU_ID_SKU_14,SKU_ID_SKU_15,SKU_ID_SKU_16,SKU_ID_SKU_17,SKU_ID_SKU_18,SKU_ID_SKU_19,SKU_ID_SKU_2,SKU_ID_SKU_20,SKU_ID_SKU_21,SKU_ID_SKU_22,SKU_ID_SKU_23,SKU_ID_SKU_24,SKU_ID_SKU_25,SKU_ID_SKU_26,SKU_ID_SKU_27,SKU_ID_SKU_28,SKU_ID_SKU_29,SKU_ID_SKU_3,SKU_ID_SKU_30,SKU_ID_SKU_31,SKU_ID_SKU_32,SKU_ID_SKU_33,SKU_ID_SKU_34,SKU_ID_SKU_35,SKU_ID_SKU_36,SKU_ID_SKU_37,SKU_ID_SKU_38,SKU_ID_SKU_39,SKU_ID_SKU_4,SKU_ID_SKU_40,SKU_ID_SKU_41,SKU_ID_SKU_42,SKU_ID_SKU_43,SKU_ID_SKU_44,SKU_ID_SKU_45,SKU_ID_SKU_46,SKU_ID_SKU_47,SKU_ID_SKU_48,SKU_ID_SKU_49,SKU_ID_SKU_5,SKU_ID_SKU_50,SKU_ID_SKU_6,SKU_ID_SKU_7,SKU_ID_SKU_8,SKU_ID_SKU_9,Warehouse_ID_WH_2,Warehouse_ID_WH_3,Warehouse_ID_WH_4,Warehouse_ID_WH_5,Supplier_ID_SUP_10,Supplier_ID_SUP_2,Supplier_ID_SUP_3,Supplier_ID_SUP_4,Supplier_ID_SUP_5,Supplier_ID_SUP_6,Supplier_ID_SUP_7,Supplier_ID_SUP_8,Supplier_ID_SUP_9,Region_North,Region_South,Region_West
0,2024-01-01,10,592,14,379,0,13.950,20.480,0,0,8.520,2024,1,1,0,1,1,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1
10585,2024-01-01,41,577,3,326,0,9.060,11.690,0,0,37.540,2024,1,1,0,1,1,10.000,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0
68255,2024-01-01,18,523,8,344,0,19.610,30.890,0,0,18.040,2024,1,1,0,1,1,41.000,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1
68620,2024-01-01,26,704,13,349,0,17.810,24.940,0,0,25.730,2024,1,1,0,1,1,18.000,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1
10220,2024-01-01,23,820,11,220,0,14.550,25.130,0,0,27.410,2024,1,1,0,1,1,26.000,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,1,0


One hot encoding for the single Product

In [ ]:
# One-hot encode the SKU_18 subset using the same encoding logic
df_sku18_fe = pd.get_dummies(df_sku18_fe, drop_first=True)

# Drop NaNs created by lag/rolling/slope windows
df_sku18_fe = df_sku18_fe.dropna().reset_index(drop=True)

# Select feature columns (exclude target)
feature_cols_sku18 = [col for col in df_sku18_fe.columns
                      if col not in ['Units_Sold']]

# Final X and y for SKU_18 modeling
X_sku18 = df_sku18_fe[feature_cols_sku18]
y_sku18 = df_sku18_fe['Units_Sold']

 - **final feature engineering steps**

In [ ]:
#Drop NaNs created by lag/rolling windows
df = df.dropna().reset_index(drop=True)


#Saving the engineered dataset to Google Drive
df.to_csv("/content/drive/MyDrive/supply_chain_thesis/engineered_dataset.csv", index=False)


#SUmmary

In Feature engineering I transformed the raw supply‑chain dataset into a structured, model‑ready version by adding information that captures time patterns, demand behavior, promotions, and product/location identity.

I created** temporal features** (year, month, weekday, week, quarter) to expose seasonality;
- **lag features** to give the model memory of past demand;
- **rolling features** to summarize recent trends and volatility; promotion‑related features to quantify marketing effects; and

- **one‑hot encoded**categorical identifiers so models can distinguish SKUs, warehouses, suppliers, and regions. Together, these features give machine‑learning models a complete view of historical patterns and operational context, enabling accurate demand forecasting.